# SPU demo4

不同采样方案下，推理的性能

## 1. 加载模型

In [1]:
import jax
from helper import load_from_cache, generate, generate_topk, generate_greedy, generate_minp

base_path = "/root/.cache/huggingface/hub/models--state-spaces--mamba-130m-hf/snapshots/1e76775f628fbf1350fbe4dbb3d971ba64af25a1"
model, params, tokenizer = load_from_cache(base_path)

print("model loaded")

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


model loaded


### 2.1: 定义运行函数

In [2]:
# 目的是加上jit
# 重要：topk非常非常慢，而且运行时间和topk的值线性相关

gen_len = 3
prompt = "Python is"
seed = 10086
topk = 40
minp = 0.1

input_ids = tokenizer.encode(prompt, return_tensors='jax')

@jax.jit
def gen_greedy(params, input_ids):
    return generate_greedy(model, params, input_ids, n_tokens_to_gen=gen_len) # 贪心采样，最快

@jax.jit
def gen_minp(params, input_ids):
    return generate_minp(model, params, input_ids, n_tokens_to_gen=gen_len, seed=seed, min_p=minp) # minp采样，比贪心慢一点点，但效果很好

@jax.jit
def gen_topk(params, input_ids):
    return generate_topk(model, params, input_ids, n_tokens_to_gen=gen_len, seed=seed, top_k=topk) # topk采样，最慢

print(f"prompt: {prompt}")
print(f"input len: {len(input_ids[0])}")
print(f"output len: {gen_len}")
print(f"seed: {seed}")
print(f"topk: {topk}")
print(f"minp: {minp}")

prompt: Python is
input len: 2
output len: 3
seed: 10086
topk: 40
minp: 0.1


### 2.2 定义模拟器

In [3]:
import sml.utils.emulation as emulation

mode = emulation.Mode.MULTIPROCESS
emulator = emulation.Emulator("3pc.json",mode)

emulator.up()

[2026-04-08 14:13:51,386]-[INFO]-[emulation.py:112]: Start multiprocess cluster...
[2026-04-08 14:13:51,982] [ForkServerProcess-1] Starting grpc server at 127.0.0.1:61920
[2026-04-08 14:13:51,983] [ForkServerProcess-3] Starting grpc server at 127.0.0.1:61922
[2026-04-08 14:13:51,988] [ForkServerProcess-5] Starting grpc server at 127.0.0.1:61924
[2026-04-08 14:13:51,989] [ForkServerProcess-4] Starting grpc server at 127.0.0.1:61923
[2026-04-08 14:13:51,995] [ForkServerProcess-2] Starting grpc server at 127.0.0.1:61921
[2026-04-08 14:13:53,469] [ForkServerProcess-1] Run : builtin_spu_init at node:0
[2026-04-08 14:13:53,469] [ForkServerProcess-2] Run : builtin_spu_init at node:1
[2026-04-08 14:13:53,469] [ForkServerProcess-3] Run : builtin_spu_init at node:2
I0408 14:13:53.486793 10476     0 external/brpc~/src/brpc/server.cpp:1195] Server[yacl::link::transport::internal::ReceiverServiceImpl] is serving on port=61932.
W0408 14:13:53.486810 10476     0 external/brpc~/src/brpc/server.cpp:120

In [4]:
@jax.jit
def warmup_fun(a, b, c):
    return a * b + c

a, b, c = 3, 2, 1
a, b, c = emulator.seal(a, b, c)
result = emulator.run(warmup_fun)(a, b, c)

result

[2026-04-08 14:13:53,634] [ForkServerProcess-4] Run : <lambda> at node:3
[2026-04-08 14:13:53,635] [ForkServerProcess-4] Run : <lambda> at node:3
[2026-04-08 14:13:53,636] [ForkServerProcess-4] Run : <lambda> at node:3
[2026-04-08 14:13:53,639] [ForkServerProcess-4] Run : make_shares at node:3
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


[2026-04-08 14:13:53.777] [info] [api.cc:172] [Profiling] SPU execution warmup_fun completed, input processing took 9.6e-07s, execution took 0.005317785s, output processing took 1.45e-06s, total time 0.005320195s.
[2026-04-08 14:13:53.777] [info] [api.cc:220] HLO profiling: total time 0.001754979
[2026-04-08 14:13:53.777] [info] [api.cc:223] - pphlo.multiply, executed 1 times, duration 0.001735519s, send bytes 16 recv bytes 16, send actions 1, recv actions 1
[2026-04-08 14:13:53.777] [info] [api.cc:223] - pphlo.add, executed 1 times, duration 1.499e-05s, send bytes 0 recv bytes 0, send actions 0, recv actions 0
[2026-04-08 14:13:53.777] [info] [api.cc:223] - pphlo.free, executed 1 times, duration 4.47e-06s, send bytes 0 recv bytes 0, send actions 0, recv actions 0
[2026-04-08 14:13:53.777] [info] [api.cc:220] HAL profiling: total time 0.000315278
[2026-04-08 14:13:53.777] [info] [api.cc:223] - i_mul, executed 1 times, duration 0.000304518s, send bytes 16 recv bytes 16, send actions 1, 

[2026-04-08 14:13:53,692] [ForkServerProcess-4] RunR: builtin_fetch_meta at node:3
[2026-04-08 14:13:53,694] [ForkServerProcess-4] Run : make_shares at node:3
[2026-04-08 14:13:53,696] [ForkServerProcess-4] RunR: builtin_fetch_meta at node:3
[2026-04-08 14:13:53,698] [ForkServerProcess-4] Run : make_shares at node:3
[2026-04-08 14:13:53,699] [ForkServerProcess-4] RunR: builtin_fetch_meta at node:3
[2026-04-08 14:13:53,766] [ForkServerProcess-1] Run : builtin_spu_run at node:0
[2026-04-08 14:13:53,766] [ForkServerProcess-3] Run : builtin_spu_run at node:2
[2026-04-08 14:13:53,766] [ForkServerProcess-2] Run : builtin_spu_run at node:1
[2026-04-08 14:13:53,768] [ForkServerProcess-4] RunR: builtin_fetch_object at node:3
[2026-04-08 14:13:53,768] [ForkServerProcess-4] RunR: builtin_fetch_object at node:3
[2026-04-08 14:13:53,769] [ForkServerProcess-4] RunR: builtin_fetch_object at node:3
[2026-04-08 14:13:53,769] [ForkServerProcess-4] RunR: builtin_fetch_object at node:3
[2026-04-08 14:13:5

array(7, dtype=int32)

### 3.1 贪心搜索


In [5]:
output_ids = gen_greedy(params, input_ids)
print(prompt, tokenizer.decode(output_ids[0]), sep='')

Python is a great tool


In [6]:
s_params, s_input_ids = emulator.seal(params, input_ids)
result = emulator.run(gen_greedy)(s_params, s_input_ids)

[2026-04-08 14:14:03,554] [ForkServerProcess-4] Run : <lambda> at node:3
[2026-04-08 14:14:03,607] [ForkServerProcess-4] Run : <lambda> at node:3
[2026-04-08 14:14:03,611] [ForkServerProcess-4] Run : make_shares at node:3


[2026-04-08 14:14:03.611] [info] [thread_pool.cc:30] Create a fixed thread pool with size 95


[2026-04-08 14:14:09,707] [ForkServerProcess-4] RunR: builtin_fetch_meta at node:3
[2026-04-08 14:14:09,710] [ForkServerProcess-4] Run : make_shares at node:3
[2026-04-08 14:14:09,714] [ForkServerProcess-4] RunR: builtin_fetch_meta at node:3
[2026-04-08 14:14:09,716] [ForkServerProcess-4] Run : make_shares at node:3
[2026-04-08 14:14:09,718] [ForkServerProcess-4] RunR: builtin_fetch_meta at node:3
[2026-04-08 14:14:09,720] [ForkServerProcess-4] Run : make_shares at node:3
[2026-04-08 14:14:09,721] [ForkServerProcess-4] RunR: builtin_fetch_meta at node:3
[2026-04-08 14:14:09,723] [ForkServerProcess-4] Run : make_shares at node:3
[2026-04-08 14:14:09,725] [ForkServerProcess-4] RunR: builtin_fetch_meta at node:3
[2026-04-08 14:14:09,728] [ForkServerProcess-4] Run : make_shares at node:3
[2026-04-08 14:14:09,730] [ForkServerProcess-4] RunR: builtin_fetch_meta at node:3
[2026-04-08 14:14:09,732] [ForkServerProcess-4] Run : make_shares at node:3
[2026-04-08 14:14:09,740] [ForkServerProcess-4

[2026-04-08 14:14:47.570] [info] [thread_pool.cc:30] Create a fixed thread pool with size 95
[2026-04-08 14:14:47.571] [info] [thread_pool.cc:30] Create a fixed thread pool with size 95
[2026-04-08 14:14:47.571] [info] [thread_pool.cc:30] Create a fixed thread pool with size 95
[2026-04-08 14:16:56.454] [info] [api.cc:172] [Profiling] SPU execution gen_greedy completed, input processing took 6.176e-05s, execution took 129.2225935s, output processing took 3.54e-06s, total time 129.2226588s.
[2026-04-08 14:16:56.502] [info] [api.cc:220] HLO profiling: total time 129.138792698
[2026-04-08 14:16:56.502] [info] [api.cc:223] - pphlo.exponential, executed 312 times, duration 42.065195215s, send bytes 4013924352 recv bytes 3726096384, send actions 22869, recv actions 21277
[2026-04-08 14:16:56.502] [info] [api.cc:223] - pphlo.convolution, executed 72 times, duration 25.916459096s, send bytes 7495680 recv bytes 7446528, send actions 410, recv actions 408
[2026-04-08 14:16:56.502] [info] [api.cc

[2026-04-08 14:16:56,800] [ForkServerProcess-1] RunR: builtin_fetch_meta at node:0
[2026-04-08 14:16:56,805] [ForkServerProcess-1] RunR: builtin_gc at node:0
[2026-04-08 14:16:56,805] [ForkServerProcess-2] RunR: builtin_gc at node:1
[2026-04-08 14:16:56,807] [ForkServerProcess-3] RunR: builtin_gc at node:2
[2026-04-08 14:16:56,807] [ForkServerProcess-5] RunR: builtin_gc at node:4
[2026-04-08 14:16:56,807] [ForkServerProcess-4] RunR: builtin_gc at node:3
[2026-04-08 14:16:56,830] [ForkServerProcess-3] RunR: builtin_gc at node:2
[2026-04-08 14:16:56,831] [ForkServerProcess-2] RunR: builtin_gc at node:1
[2026-04-08 14:16:56,831] [ForkServerProcess-1] RunR: builtin_gc at node:0
[2026-04-08 14:16:56,831] [ForkServerProcess-5] RunR: builtin_gc at node:4
[2026-04-08 14:16:56,832] [ForkServerProcess-4] RunR: builtin_gc at node:3
[2026-04-08 14:16:56,878] [ForkServerProcess-2] RunR: builtin_gc at node:1
[2026-04-08 14:16:56,878] [ForkServerProcess-1] RunR: builtin_gc at node:0
[2026-04-08 14:16

In [7]:
print(result)
print(prompt, tokenizer.decode(result[0]), sep='')

[[ 247 1270 4968]]
Python is a great tool


### 3.2 minp采样

In [8]:
output_ids = gen_minp(params, input_ids)
print(prompt, tokenizer.decode(output_ids[0]), sep='')

Python is not required.


In [9]:
s_params, s_input_ids = emulator.seal(params, input_ids)
result = emulator.run(gen_minp)(s_params, s_input_ids)

[2026-04-08 14:17:05,414] [ForkServerProcess-4] Run : <lambda> at node:3
[2026-04-08 14:17:05,519] [ForkServerProcess-4] Run : <lambda> at node:3
[2026-04-08 14:17:05,523] [ForkServerProcess-1] RunR: builtin_gc at node:0
[2026-04-08 14:17:05,525] [ForkServerProcess-4] RunR: builtin_gc at node:3
[2026-04-08 14:17:05,525] [ForkServerProcess-3] RunR: builtin_gc at node:2
[2026-04-08 14:17:05,525] [ForkServerProcess-2] RunR: builtin_gc at node:1
[2026-04-08 14:17:05,526] [ForkServerProcess-5] RunR: builtin_gc at node:4
[2026-04-08 14:17:05,530] [ForkServerProcess-2] RunR: builtin_gc at node:1
[2026-04-08 14:17:05,530] [ForkServerProcess-1] RunR: builtin_gc at node:0
[2026-04-08 14:17:05,531] [ForkServerProcess-3] RunR: builtin_gc at node:2
[2026-04-08 14:17:05,531] [ForkServerProcess-4] RunR: builtin_gc at node:3
[2026-04-08 14:17:05,532] [ForkServerProcess-5] RunR: builtin_gc at node:4
[2026-04-08 14:17:05,536] [ForkServerProcess-2] RunR: builtin_gc at node:1
[2026-04-08 14:17:05,536] [Fo

[2026-04-08 14:19:56.048] [info] [api.cc:172] [Profiling] SPU execution gen_minp completed, input processing took 0.000248588s, execution took 127.701575642s, output processing took 2.23e-06s, total time 127.70182646s.
[2026-04-08 14:19:56.099] [info] [api.cc:220] HLO profiling: total time 127.612601057
[2026-04-08 14:19:56.099] [info] [api.cc:223] - pphlo.exponential, executed 312 times, duration 41.602587784s, send bytes 4011687936 recv bytes 3757565952, send actions 22851, recv actions 21335
[2026-04-08 14:19:56.099] [info] [api.cc:223] - pphlo.convolution, executed 72 times, duration 25.631752722s, send bytes 7274496 recv bytes 7127040, send actions 400, recv actions 387
[2026-04-08 14:19:56.099] [info] [api.cc:223] - pphlo.dot_general, executed 48 times, duration 21.713472156s, send bytes 80179200 recv bytes 78769920, send actions 98356, recv actions 98220
[2026-04-08 14:19:56.099] [info] [api.cc:223] - pphlo.dot, executed 291 times, duration 17.341866058s, send bytes 24545536 rec

[2026-04-08 14:19:56,343] [ForkServerProcess-1] RunR: builtin_fetch_meta at node:0
[2026-04-08 14:19:56,349] [ForkServerProcess-1] RunR: builtin_gc at node:0
[2026-04-08 14:19:56,349] [ForkServerProcess-2] RunR: builtin_gc at node:1
[2026-04-08 14:19:56,349] [ForkServerProcess-4] RunR: builtin_gc at node:3
[2026-04-08 14:19:56,349] [ForkServerProcess-3] RunR: builtin_gc at node:2
[2026-04-08 14:19:56,349] [ForkServerProcess-5] RunR: builtin_gc at node:4
[2026-04-08 14:19:56,354] [ForkServerProcess-1] RunR: builtin_gc at node:0
[2026-04-08 14:19:56,356] [ForkServerProcess-2] RunR: builtin_gc at node:1
[2026-04-08 14:19:56,357] [ForkServerProcess-4] RunR: builtin_gc at node:3
[2026-04-08 14:19:56,357] [ForkServerProcess-3] RunR: builtin_gc at node:2
[2026-04-08 14:19:56,357] [ForkServerProcess-5] RunR: builtin_gc at node:4
[2026-04-08 14:19:56,391] [ForkServerProcess-1] RunR: builtin_gc at node:0
[2026-04-08 14:19:56,392] [ForkServerProcess-3] RunR: builtin_gc at node:2
[2026-04-08 14:19

In [10]:
print(result)
print(prompt, tokenizer.decode(result[0]), sep='')

[[ 247 1270 4968]]
Python is a great tool


### 3.3 topk采样

In [11]:
output_ids = gen_topk(params, input_ids)
print(prompt, tokenizer.decode(output_ids[0]), sep='')

Python is not required.


In [12]:
s_params, s_input_ids = emulator.seal(params, input_ids)
result = emulator.run(gen_topk)(s_params, s_input_ids)

[2026-04-08 14:20:05,385] [ForkServerProcess-4] Run : <lambda> at node:3
[2026-04-08 14:20:05,454] [ForkServerProcess-4] Run : <lambda> at node:3
[2026-04-08 14:20:05,459] [ForkServerProcess-3] RunR: builtin_gc at node:2
[2026-04-08 14:20:05,459] [ForkServerProcess-2] RunR: builtin_gc at node:1
[2026-04-08 14:20:05,459] [ForkServerProcess-5] RunR: builtin_gc at node:4
[2026-04-08 14:20:05,459] [ForkServerProcess-1] RunR: builtin_gc at node:0
[2026-04-08 14:20:05,460] [ForkServerProcess-4] RunR: builtin_gc at node:3
[2026-04-08 14:20:05,722] [ForkServerProcess-1] RunR: builtin_gc at node:0
[2026-04-08 14:20:05,723] [ForkServerProcess-3] RunR: builtin_gc at node:2
[2026-04-08 14:20:05,723] [ForkServerProcess-2] RunR: builtin_gc at node:1
[2026-04-08 14:20:05,723] [ForkServerProcess-4] RunR: builtin_gc at node:3
[2026-04-08 14:20:05,724] [ForkServerProcess-5] RunR: builtin_gc at node:4
[2026-04-08 14:20:05,730] [ForkServerProcess-1] RunR: builtin_gc at node:0
[2026-04-08 14:20:05,730] [Fo

[2026-04-08 14:22:51.372] [info] [api.cc:172] [Profiling] SPU execution gen_topk completed, input processing took 0.000332288s, execution took 126.781372058s, output processing took 2.02e-06s, total time 126.781706366s.
[2026-04-08 14:22:51.399] [info] [api.cc:220] HLO profiling: total time 126.68777074399999
[2026-04-08 14:22:51.399] [info] [api.cc:223] - pphlo.exponential, executed 312 times, duration 41.849414306s, send bytes 4020854784 recv bytes 3736694784, send actions 22850, recv actions 21257
[2026-04-08 14:22:51.399] [info] [api.cc:223] - pphlo.convolution, executed 72 times, duration 25.527065888s, send bytes 7421952 recv bytes 7495680, send actions 405, recv actions 407
[2026-04-08 14:22:51.399] [info] [api.cc:223] - pphlo.dot_general, executed 48 times, duration 21.853159126s, send bytes 81011712 recv bytes 79614208, send actions 98448, recv actions 98358
[2026-04-08 14:22:51.399] [info] [api.cc:223] - pphlo.dot, executed 291 times, duration 15.437910766s, send bytes 261506

[2026-04-08 14:22:51,584] [ForkServerProcess-1] RunR: builtin_fetch_meta at node:0
[2026-04-08 14:22:51,589] [ForkServerProcess-2] RunR: builtin_gc at node:1
[2026-04-08 14:22:51,589] [ForkServerProcess-3] RunR: builtin_gc at node:2
[2026-04-08 14:22:51,589] [ForkServerProcess-1] RunR: builtin_gc at node:0
[2026-04-08 14:22:51,590] [ForkServerProcess-4] RunR: builtin_gc at node:3
[2026-04-08 14:22:51,590] [ForkServerProcess-5] RunR: builtin_gc at node:4
[2026-04-08 14:22:51,612] [ForkServerProcess-2] RunR: builtin_gc at node:1
[2026-04-08 14:22:51,612] [ForkServerProcess-1] RunR: builtin_gc at node:0
[2026-04-08 14:22:51,612] [ForkServerProcess-5] RunR: builtin_gc at node:4
[2026-04-08 14:22:51,612] [ForkServerProcess-3] RunR: builtin_gc at node:2
[2026-04-08 14:22:51,612] [ForkServerProcess-4] RunR: builtin_gc at node:3
[2026-04-08 14:22:51,634] [ForkServerProcess-1] RunR: builtin_gc at node:0
[2026-04-08 14:22:51,634] [ForkServerProcess-4] RunR: builtin_gc at node:3
[2026-04-08 14:22

In [13]:
print(result)
print(prompt, tokenizer.decode(result[0]), sep='')

[[ 247 1270 4968]]
Python is a great tool


## 4. 停止模拟器

In [14]:
emulator.down()

[2026-04-08 14:22:52,190]-[INFO]-[emulation.py:120]: Shutdown multiprocess cluster...
